In [318]:
import awswrangler as wr

In [319]:
def exec_sql(sql_statement:str, db="financials"):
  return wr.athena.read_sql_query(
        sql=sql_statement,
        database=db
    )

In [321]:
sql ="""
    SELECT *
    FROM silver_t212_positions
    ORDER BY ingested_date DESC
    limit 10;
"""

df_positions = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


In [246]:
sql ="""
    WITH account_summary AS (
      SELECT 
          ROW_NUMBER() OVER (PARTITION BY ingested_date ORDER BY ingested_timestamp DESC) as rn
        , total_value_investmented
      FROM silver_t212_account_summary
    limit 10
    )
    SELECT *
    FROM cte
    WHERE rn = 1
    ORDER BY ingested_date DESC
"""

df_account_summary = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


QueryFailed: TABLE_NOT_FOUND: line 16:3: Table 'awsdatacatalog.financials.cte' does not exist. You may need to manually clean the data at location 's3://aws-athena-query-results-861580917950-eu-west-1/tables/c6066fa5-3bec-4011-95f2-c1a9da662aea' before retrying. Athena will not delete data in your account.

In [97]:
df_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   name                            10 non-null     string        
 1   ticker                          10 non-null     string        
 2   isin                            10 non-null     string        
 3   created_at                      10 non-null     datetime64[ns]
 4   asset_currency                  10 non-null     string        
 5   avg_price_paid                  10 non-null     float64       
 6   current_price                   10 non-null     float64       
 7   quantity                        10 non-null     float64       
 8   quantity_available_for_trading  10 non-null     float64       
 9   quantity_in_pies                10 non-null     float64       
 10  account_currency                10 non-null     string        
 11  current_value       

In [ ]:
from datetime import datetime, timezone

current_date = datetime.now(timezone.utc).date()

sql =F"""
SELECT name
, ticker
, weight_pct
, daily_value_change_pct
, current_value
, avg_value_7d
, pnl
,ingested_timestamp
FROM fact_t212_positions
WHERE
    ingested_date = date_parse('{current_date}', '%Y-%m-%d')
  AND daily_value_change_pct <= -1
  AND current_value < avg_value_7d 
ORDER BY ingested_timestamp, daily_value_change_pct, weight_pct
"""

df_fact = exec_sql(sql)

df

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,name,ticker,weight_pct,daily_value_change_pct,current_value,avg_value_7d,pnl,ingested_timestamp,top_up,target_value,rebalance_amount
4,Chevron,CHVd_EQ,0.863769,-1.404572,1.036523,72.037143,12.60,2026-09-25 14:35:45.129,6.163477,7.2,6.163477
11,Invesco Energy S&P US Select Sector (Acc),XLEPl_EQ,5.287571,-2.296032,6.345086,445.702857,42.73,2026-09-25 14:35:45.129,0.854914,7.2,0.854914
19,British American Tobacco,BMT1d_EQ,0.606569,-1.527615,0.727882,50.594286,0.10,2026-09-25 14:35:45.129,6.472118,7.2,6.472118


In [307]:
# x target allocation is 6
# equity 100

equity = 120
target_weight = 6.00
current_weight = 5.287571	

df["top_up"] = equity * (abs(target_weight - df["weight_pct"])) / 100

df

,name,ticker,weight_pct,daily_value_change_pct,current_value,avg_value_7d,pnl,ingested_timestamp,top_up,target_value,rebalance_amount
4,Chevron,CHVd_EQ,0.863769,-1.404572,1.036523,72.037143,12.60,2026-09-25 14:35:45.129,6.163477,6.0,4.963477
11,Invesco Energy S&P US Select Sector (Acc),XLEPl_EQ,5.287571,-2.296032,6.345086,445.702857,42.73,2026-09-25 14:35:45.129,0.854914,6.0,-0.345086
19,British American Tobacco,BMT1d_EQ,0.606569,-1.527615,0.727882,50.594286,0.10,2026-09-25 14:35:45.129,6.472118,6.0,5.272118


In [308]:
# Calculate dollar values
df["target_value"] = equity * (target_weight / 100)
df["current_value"] = equity * (df["weight_pct"] / 100)
# Positive = Need to BUY, Negative = Need to SELL
df["rebalance_amount"] = df["target_value"] - df["current_value"]
print(df[["ticker", "current_value", "weight_pct", "target_value", "rebalance_amount"]])

      ticker  current_value  weight_pct  target_value  rebalance_amount
4    CHVd_EQ       1.036523    0.863769           7.2          6.163477
11  XLEPl_EQ       6.345086    5.287571           7.2          0.854914
19  BMT1d_EQ       0.727882    0.606569           7.2          6.472118


In [269]:
(target_weight - current_weight) / 100


0.007124290000000002

In [263]:
6/100 * 100

6.0

In [ ]:
from_date = "2026-08-17"
to_date = "2026-09-25"

sql =F"""
WITH asset_mapping AS (
  SELECT *
  FROM
  asset_mapping
  
)
MERGE INTO t212_asset_dim t
  USING silver_t212_positions s
  ON t.ticker = s.ticker
    AND t.name = s.name
    
  
WHEN MATCHED THEN
  UPDATE SET
  
"""

t212_asset_dim = exec_sql(sql)

In [338]:
x = f"""SELECT name
, ticker
, weight_pct
, daily_value_change_pct
, current_value
, avg_value_7d
, pnl
,ingested_timestamp
FROM fact_t212_positions
"""

x = exec(x)

SyntaxError: invalid syntax (<string>, line 1)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


In [378]:
df

,name,ticker,weight_pct,daily_value_change_pct,current_value,avg_value_7d,pnl,ingested_timestamp
0,Vanguard FTSE All-World (Acc),VWRPl_EQ,17.309866,0.21231,1434.91,1430.732857,94.42,2026-09-27 15:30:32.631


In [ ]:
dynamo_db = boto3.client("dynamodb")

In [408]:
trade_logs

""


In [403]:
from uuid import uuid4

In [410]:
str(uuid4())

'4c87e119-c5c7-4120-a2be-be707e813bb1'

In [412]:

import logging

logger = logging.getLogger()

logger.setLevel(level="INFO")

In [417]:
import httpx
from httpx import ClientError

ImportError: cannot import name 'ClientError' from 'httpx' (/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/httpx/__init__.py)

In [ ]:
import json
import boto3
from botocore.exceptions import ClientError
from datetime import datetime, timezone
from uuid import uuid4
import logging
import httpx

logger = logging.getLogger()

logger.setLevel(level="INFO")


database = "financials"
trade_log_table_name = "trade_log"


SECRET_TOKEN = None
API_TOKEN = None

trade_logs = None


domain = ""
endpoint = "equity/orders/market"

today = datetime.now(timezone.utc)
ticker = "VWRPl_EQ"
equity = 15
max_drawdown_threshold = -1
min_drawdown_threshold = -2.5

try:
  logger.info("Fetching secrets")
  secrets_manager = boto3.client("secretsmanager")
  response = secrets_manager.get_secret_value(
      SecretId='prod/financial-dataflow/trading212',
  )
  secret = json.loads (response.get("SecretString"))
  SECRET_TOKEN = secret.get("T212_SECRET_TOKEN")
  API_TOKEN = secret.get("T212_API_TOKEN")
  logger.info("Fetched secrets")
except ClientError as e:
  logger.error(e)
  
  
if SECRET_TOKEN is None or API_TOKEN is None:
  logger.warning("Missing credentials, exiting appliction")
  print("return")
  
try:
  logger.info("Fetching trade logs")
  trade_logs = wr.dynamodb.read_items(
    table_name=trade_log_table_name
    , partition_values=['trade_id']
    , consistent=True
    , sort_values=["created_datetime"]
    , max_items_evaluated=10)
  logger.info("Fetched trade logs")
except Exception as e:
  logger.error(e)


sql = F"""
SELECT name
, ticker
, weight_pct
, daily_value_change_pct
, current_value
, avg_value_7d
, pnl
,ingested_timestamp
FROM fact_t212_positions
WHERE ticker = '{ticker}'
ORDER BY ingested_date DESC
LIMIT 1;
"""


try:
  logger.info("Fetching Asset data")
  wr.athena.read_sql_query(
      sql=sql,
      database=database
  )
  logger.info("Fetched Asset data")
except Exception as e:
  logger.erro(e)
  

change_pct = df["daily_value_change_pct"].iloc[0]
avg_value_7d = df["avg_value_7d"].iloc[0]
value = df["current_value"].iloc[0]


if not (max_drawdown_threshold <= change_pct and min_drawdown_threshold >= change_pct):
  print("return")
  
if value => avg_value_7d:
    print("return")


credentials = f"{api_token}:{secret_token}"
token = base64.b64encode(credentials.encode("utf-8")).decode("utf-8")

header = {"Authorization": f"Basic {token}"}
payload = {"ticker": ticker,"value": equity}

try:
  logger.info("Entering market order - Asset: %s, Value: %i", [ticker, equity])
  with httpx.Client() as client:
    url = urljoin(domain, endpoint)
    response = client.post(url, json=payload, header=header, timeout=10)
    response.raise_for_status()
  logger.info("Entered Trade")
except httpx.HTTPError as e:
  logger.error(e)
  
  
  
trade = {
  "id": str(uuid4()),
  "vendor": "trading-212",
  "asset": ticker,
  "value": payload.get("value")
  "quantity": "",
  "trader": "T212 DCA Automation"
  "created_datetime": datetime.now(timezone.utc),
}

try:
  logger.info("Writing trade log")
  wr.dynamodb.put_items(table_name=trade_log_table_name, items=[trade])
except Exception e:
  logger.error(e)

In [411]:
datetime.now(timezone.utc)

datetime.datetime(2026, 9, 27, 22, 24, 18, 437710, tzinfo=datetime.timezone.utc)

In [ ]:
df["daily_value_change_pct"].iloc[0]

np.float64(0.21230977672555407)